# Muonet quickstart

**Cosmic rays in, 3D interiors out.** This notebook runs the whole pipeline end to end on a free Colab CPU: simulate cosmic muons scattering through a dense object, reconstruct the 3D interior with the classical PoCA baseline, and watch reconstruction quality climb as more muons arrive.

Everything here is deterministic from a seed and needs no GPU, no download, and no account.

Repo: https://github.com/KarthikSubramanian07/muonet

In [ ]:
# Torch ships with Colab; this installs Muonet and its light dependencies.
!pip install -q git+https://github.com/KarthikSubramanian07/muonet.git

## 1. Build a scene and simulate muons

A 10 cm tungsten cube sits in air inside a 32 cm voxel grid, sandwiched by two tracking stations. We fire cosmic muons through it under a realistic (imperfect) detector.

In [ ]:
import numpy as np
from muonet.scenes import build_scene, default_grid, default_detector
from muonet.detector import get_detector_config
from muonet.data import collect_measurement
from muonet.recon import poca_reconstruct
from muonet.metrics import evaluate

grid = default_grid()
detector = default_detector()
scene = build_scene('tungsten_cube', grid)
config = get_detector_config('realistic_sipm')

rng = np.random.default_rng(0)
meas = collect_measurement(scene, detector, config, 4096, rng)
print('reconstructable muons:', len(meas))

## 2. Reconstruct with PoCA and score it

The reconstructed quantity is scattering density (inverse radiation length), which is what muon scattering physically measures.

In [ ]:
volume = poca_reconstruct(meas, grid)
metrics = evaluate(volume, scene.normalized_density(), scene.high_z_mask())
print({k: round(v, 3) for k, v in metrics.items()})

In [ ]:
import matplotlib.pyplot as plt
from muonet.metrics import normalize_volume

gt = scene.normalized_density()
recon = normalize_volume(volume)
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(gt[:, :, 16].T, origin='lower', cmap='inferno'); ax[0].set_title('ground truth')
ax[1].imshow(recon[:, :, 16].T, origin='lower', cmap='inferno'); ax[1].set_title('PoCA reconstruction')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 3. The dosage curve

This is the core physics: more muons means more information. Watch PSNR and high-Z IoU climb with muon count.

In [ ]:
dosages = [128, 256, 512, 1024, 4096, 8192]
psnr, iou = [], []
for d in dosages:
    rng = np.random.default_rng(0)
    m = collect_measurement(scene, detector, config, d, rng)
    r = evaluate(poca_reconstruct(m, grid), scene.normalized_density(), scene.high_z_mask())
    psnr.append(r['psnr']); iou.append(r['iou'])

fig, ax1 = plt.subplots(figsize=(6, 4))
ax1.plot(dosages, psnr, 'o-', color='#4c6ef5', label='PSNR')
ax1.set_xscale('log', base=2); ax1.set_xlabel('muons'); ax1.set_ylabel('PSNR (dB)')
ax2 = ax1.twinx(); ax2.plot(dosages, iou, 's-', color='#e8590c', label='IoU'); ax2.set_ylabel('high-Z IoU')
ax1.set_title('PoCA reconstruction quality vs muon dosage')
plt.tight_layout(); plt.show()

## 4. Train the learned model (optional, a few minutes on CPU)

The learned two-stage model (trajectory MLP + compact 3D U-Net) is a fast quick-train here. On a free Colab T4 it is quicker still. This architecture family is from prior work; Muonet's contribution is the reproducible, detector-realistic package around it.

In [ ]:
from muonet.train import TrainConfig, build_learned

cfg = TrainConfig(device='cpu')
cfg.unet_scenes, cfg.unet_epochs = 48, 12  # quick demo; defaults are larger
reconstructor, stats = build_learned(cfg, seed=0, verbose=True)
print('stage-1 track error (mrad):', {k: round(v, 2) for k, v in stats.items()})

learned_vol = reconstructor.reconstruct(meas, grid)
print('PoCA   ', {k: round(v, 3) for k, v in evaluate(volume, gt, scene.high_z_mask()).items()})
print('Learned', {k: round(v, 3) for k, v in evaluate(learned_vol, gt, scene.high_z_mask()).items()})

## Where to go next

- Reproduce every published number: `muonet bench --checkpoint checkpoints/muonet.pt`
- Describe your own geometry in a YAML config and reconstruct it: `muonet recon --config your_scene.yaml --method poca`
- Export volumes for the interactive 3D viewer: `muonet export`

Honest scope: Muonet reconstructs simple, single-grid geometries with real detector physics and reproducible benchmarks. It is a research and education toolkit, not a certified scanner.